# Create Three Continual-Learning Stage Files

Selected stages:

- Stage 1: 2010–2018
- Stage 2: 2019–2022
- Stage 3: 2023

For every stage:

1. A chronological holdout is taken from the beginning.
2. A 24-hour NF-only gap separates the holdout from training.
3. Holdout samples are not augmented or used for training.
4. Training and holdout files contain only:
   - `label`
   - `goes_class`

NF-only gaps are also removed between consecutive stages.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the repository root.
possible_roots = [Path.cwd(), *Path.cwd().parents]

ROOT = next(
    path for path in possible_roots
    if (path / "data_labeling").is_dir()
)

LABEL_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "simplified_data_labels"
)

OLD_LABEL_FILE = LABEL_DIR / "labels_2010_2018_binary.csv"
NEW_LABEL_FILE = LABEL_DIR / "labels_2019_2026_july_binary.csv"

# Use a new directory so the earlier stage files are preserved.
OUTPUT_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "continual_stages_2010_2023_candidate_A"
)

print("Repository:", ROOT)
print("Output directory:", OUTPUT_DIR)

assert OLD_LABEL_FILE.is_file()
assert NEW_LABEL_FILE.is_file()

Repository: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-
Output directory: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/continual_stages_2010_2023_candidate_A


In [2]:
old_labels = pd.read_csv(OLD_LABEL_FILE)
new_labels = pd.read_csv(NEW_LABEL_FILE)

df = pd.concat(
    [old_labels, new_labels],
    ignore_index=True,
)

In [3]:
df

,label,goes_class
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0
...,...,...
128088,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,0
128089,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,0
128090,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,0
128091,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,0


In [4]:
# Convert the target into binary form.
target_mapping = {
    "NF": 0,
    "FL": 1,
    "0": 0,
    "1": 1,
    0: 0,
    1: 1,
}

df["target"] = df["goes_class"].map(target_mapping)

In [5]:
df

,label,goes_class,target
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,0,0
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,0,0
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,0,0
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,0,0
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,0,0
...,...,...,...
128088,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,0,0
128089,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,0,0
128090,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,0,0
128091,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,0,0


In [6]:
# Extract timestamps from image paths.
timestamp_parts = df["label"].str.extract(
    r"HMI\.m(?P<date>\d{4}\.\d{2}\.\d{2})_"
    r"(?P<time>\d{2}\.\d{2}\.\d{2})"
)

df["timestamp"] = pd.to_datetime(
    timestamp_parts["date"] + " " + timestamp_parts["time"],
    format="%Y.%m.%d %H.%M.%S",
    errors="coerce",
)

df["year"] = df["timestamp"].dt.year

# Only use 2010–2023 here.
stage_source = (
    df[df["year"].between(2010, 2023)]
    .dropna(subset=["timestamp", "target"])
    .copy()
)

In [7]:
stage_source["target"] = stage_source["target"].astype(int)
stage_source["goes_class"] = stage_source["target"]

stage_source = (
    stage_source
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Rows:", len(stage_source))
print("First timestamp:", stage_source["timestamp"].min())
print("Last timestamp:", stage_source["timestamp"].max())
print("Duplicate paths:", stage_source["label"].duplicated().sum())
print("Duplicate timestamps:", stage_source["timestamp"].duplicated().sum())

Rows: 106093
First timestamp: 2010-12-06 07:00:00
Last timestamp: 2023-12-31 23:00:00
Duplicate paths: 0
Duplicate timestamps: 0


In [8]:
STAGE_PERIODS = {
    1: (2010, 2018),
    2: (2019, 2022),
    3: (2023, 2023),
}

HOLDOUT_FRACTION = 0.15
GAP_HOURS = 24

# Require enough FL images to calculate useful holdout metrics.
MIN_HOLDOUT_FL = 30
MIN_TRAIN_FL = 100

print("Selected stages:")

for stage_number, years in STAGE_PERIODS.items():
    print(
        f"Stage {stage_number}: "
        f"{years[0]}–{years[1]}"
    )

Selected stages:
Stage 1: 2010–2018
Stage 2: 2019–2022
Stage 3: 2023–2023


In [9]:
def inspect_gap(data, gap_start, gap_hours=24):
    """
    Check whether a period contains exactly one image per hour
    and whether every image is NF.
    """

    gap_start = pd.Timestamp(gap_start)
    gap_end = gap_start + pd.Timedelta(
        hours=gap_hours - 1
    )

    expected_times = pd.date_range(
        gap_start,
        gap_end,
        freq="h",
    )

    selected = data[
        data["timestamp"].between(
            gap_start,
            gap_end,
        )
    ].copy()

    actual_times = pd.DatetimeIndex(
        selected["timestamp"].sort_values()
    )

    complete = (
        len(selected) == gap_hours
        and actual_times.equals(expected_times)
    )

    flare_count = int(selected["target"].sum())

    return {
        "gap_start": gap_start,
        "gap_end": gap_end,
        "rows": len(selected),
        "FL": flare_count,
        "NF": len(selected) - flare_count,
        "complete": complete,
        "valid_nf_gap": complete and flare_count == 0,
    }

In [10]:
def find_boundary_gap(data, boundary):
    """
    Find an NF-only 24-hour gap immediately before or
    immediately after a stage boundary.
    """

    boundary = pd.Timestamp(boundary)

    possible_starts = [
        boundary - pd.Timedelta(hours=GAP_HOURS),
        boundary,
    ]

    results = [
        inspect_gap(
            data,
            start,
            gap_hours=GAP_HOURS,
        )
        for start in possible_starts
    ]

    valid_results = [
        result
        for result in results
        if result["valid_nf_gap"]
    ]

    if not valid_results:
        raise RuntimeError(
            f"No complete NF-only {GAP_HOURS}-hour gap "
            f"was found next to {boundary}."
        )

    # Prefer the earlier-side gap when both are valid.
    return valid_results[0]


stage_boundaries = [
    pd.Timestamp("2019-01-01 00:00:00"),
    pd.Timestamp("2023-01-01 00:00:00"),
]

boundary_gaps = [
    find_boundary_gap(stage_source, boundary)
    for boundary in stage_boundaries
]

boundary_gap_table = pd.DataFrame(boundary_gaps)

display(boundary_gap_table)

,gap_start,gap_end,rows,FL,NF,complete,valid_nf_gap
0,2019-01-01,2019-01-01 23:00:00,24,0,24,True,True
1,2022-12-31,2022-12-31 23:00:00,24,0,24,True,True
